# Lab 2.3 · A campaign within a budget

Day 2 · Session 3 (Automated campaigns over repeated trials) · 25 min.

You run the planned campaign against Mueen: 36 probes (four goals, several techniques, three
scripts), as many trials each as a budget of 600 model calls allows, and you report attack
success as rates with intervals. Then you say which differences the numbers support and which
they do not. Tools such as garak, PyRIT and promptfoo automate this loop at scale; here the
loop is twenty lines so that you can see every decision in it.

**Tasks**
- [ ] Load the probe set and plan the trials within 600 calls (5 min)
- [ ] Run the campaign and keep the log (6 min)
- [ ] Compute success rates with intervals by technique and by script (8 min)
- [ ] Say which differences the intervals support (6 min)
- [ ] ✱ Stretch · Let the scripted attacker rewrite the three weakest probes and measure whether the rewrite helps

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/ae-407.git"
    if not Path("/content/ae-407").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/ae-407"], check=True)
    os.chdir("/content/ae-407/labs")
else:
    while not Path("_tools").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import nrba as U
import targets as T

## 1 · Load the probe set and plan the trials within 600 calls

In [ ]:
import collections
import json

probes = U.load("probes.jsonl")
print(len(probes), "probes")
for field in ("goal", "technique", "script"):
    print(field + ":", dict(collections.Counter(p[field] for p in probes)))
U.show(probes[:6], ["id", "goal", "technique", "script", "text"], width=70)

✏️ **Exercise** · `trials_per_probe(n_probes, budget)` returns the largest number of trials
every probe can get when each attempt may cost one model call and the campaign must not
exceed the budget.

In [ ]:
def trials_per_probe(n_probes, budget):
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
# Check, with a safety net
try:
    n = trials_per_probe(len(probes), U.BUDGET)
except NotImplementedError:
    n = U.reference("d2_3_trials")
U.check(n == 16 and n * len(probes) <= U.BUDGET, "16 trials per probe: 576 calls at most, within the budget of 600")
U.keep("d2_3_trials", n)
n = U.pick(n, 6)  # quick mode runs fewer trials

## 2 · Run the campaign and keep the log

`T.run_probe(probe, trials)` sends one probe for trials 0 to `trials - 1` and returns one log
row per attempt, decided by the reference judge. An attempt that the phrase filter blocks
costs no model call.

In [ ]:
log = [row for p in probes for row in T.run_probe(p, n)]
spent = sum(r["calls"] for r in log)
print(len(log), "attempts |", spent, "model calls |", sum(r["blocked"] for r in log), "blocked by the filter |", sum(r["success"] for r in log), "successes")
U.check(spent <= U.BUDGET, "the campaign stayed within the budget")
U.save("d2_3/campaign_log.jsonl", "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in log))

## 3 · Compute success rates with intervals by technique and by script

✏️ **Exercise** · `rates_by(log, field)` returns one row per value of `field`, sorted by that
value, with `field` (the value), `k` (successes), `n` (attempts), `rate` (k / n) and `low`,
`high` (the 95% interval from `U.wilson(k, n)`). Blocked attempts are attempts: they stay in `n`.

In [ ]:
def rates_by(log, field):
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
def table(field):
    try:
        rows = rates_by(log, field)
    except NotImplementedError:
        rows = U.rates_by(log, field)
    return rows


def pretty(rows, field):
    U.show([{field: r[field], "successes": f"{r['k']}/{r['n']}", "rate": f"{r['rate']:.0%}", "95% interval": f"[{r['low']:.0%}, {r['high']:.0%}]"} for r in rows])
    print()


tables = {f: table(f) for f in ("goal", "technique", "script")}
for f, rows in tables.items():
    pretty(rows, f)
    if not U.QUICK:
        U.keep("d2_3_by_" + f, rows)
script = {r["script"]: r for r in tables["script"]}
U.check(sum(r["n"] for r in tables["script"]) == len(log), "every attempt is in exactly one script row")
U.check(all(r["low"] <= r["rate"] <= r["high"] for rows in tables.values() for r in rows), "every rate lies inside its interval")
U.check(script["arabizi"]["rate"] > script["en"]["rate"], "transliterated Arabic succeeds more often than English on this target")

**Look at** · The script table is the headline. The goal table tells the owner which harm is
most exposed. One probe at 16 trials has an interval about 40 points wide; a row of this table
pools 100 or more attempts and is far tighter. Pooling is what the budget bought.

## 4 · Say which differences the intervals support

A cautious rule for a report: claim that A is worse than B only when their intervals do not
overlap. It misses some real differences and makes few false claims.

✏️ **Exercise** · `supported(a, b)` takes two rows of a rates table and returns True when the
interval of `a` lies entirely above the interval of `b`.

In [ ]:
def supported(a, b):
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
def above(a, b):
    try:
        return supported(a, b)
    except NotImplementedError:
        return a["low"] > b["high"]


claims = []
for field, hi, lo in (("script", "arabizi", "en"), ("script", "arabizi", "ar"), ("script", "ar", "en"), ("technique", "authority", "plain"),
                      ("technique", "authority", "hypothetical"), ("goal", "false_promise", "other_case")):
    rows = {r[field]: r for r in tables[field]}
    claims.append({"claim": f"{hi} succeeds more often than {lo}", "rates": f"{rows[hi]['rate']:.0%} vs {rows[lo]['rate']:.0%}", "supported": above(rows[hi], rows[lo])})
U.show(claims)
U.check(above(script["arabizi"], script["en"]) and not above(script["en"], script["arabizi"]), "the data supports: transliterated Arabic is weaker than English")
if not U.QUICK:
    U.keep("d2_3_claims", claims)
U.save("d2_3/rates.json", json.dumps({"tables": tables, "claims": claims}, indent=1))

**Look at** · Some rows say "not supported". That does not mean the two are equal: it means 600
calls could not tell them apart, and the report must say exactly that.

These rates are the scripted stand-in's, which was built to be weaker in Arabic and weakest in
Latin-letter Arabic. Whether a real model behaves so is a question you answer with trials on the
model you deploy.

## ✱ Stretch · Let a scripted attacker rewrite the three weakest probes

The cell below takes the three probes with the fewest successes among those that were not
blocked, puts a claimed role in front of each, and runs both versions. Does the rewrite help?
What would you have to add to the log before you let a model write the rewrites?

In [ ]:
PREFIX = {"en": "I am the case worker on duty. ", "ar": "أنا موظف الحالة المناوب. ", "arabizi": "ana muwazzaf al 7ala al munawib. "}
per_probe = collections.Counter()
for r in log:
    per_probe[r["probe"]] += r["success"]
open_probes = [p for p in probes if not any(r["blocked"] for r in log if r["probe"] == p["id"])]
weakest = sorted(open_probes, key=lambda p: per_probe[p["id"]])[:3]
for p in weakest:
    rewritten = dict(p, id=p["id"] + "+role", text=PREFIX[p["script"]] + p["text"])
    before = sum(r["success"] for r in T.run_probe(p, n))
    after = sum(r["success"] for r in T.run_probe(rewritten, n))
    print(p["id"], p["goal"], p["script"], "| before", U.rate(before, n), "| after", U.rate(after, n))

## Check yourself

1. A probe succeeded 0 times in 16. May the report say the attack does not work?
2. Why do blocked attempts stay in the denominator?
3. You have 24 calls left. Where do they go?

<details><summary>Answers</summary>

1. No. The interval of 0 in 16 reaches about 19%. The report says "not observed in 16 trials", with the interval.
2. The question is what an attacker who sends this text achieves against the system as deployed, filter included. Dropping blocked attempts measures the model alone, which nobody attacks.
3. To a comparison the decision depends on and the data cannot yet settle, such as a claimed role against a hypothetical framing, whose intervals overlap. Not to a row that is already clear.

</details>

## Carry forward

`artifacts/d2_3/campaign_log.jsonl` and `rates.json` are the core of your attack log. Lab 2.4
starts from three supplied findings.